In [1]:
# 1. Imports & Setup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

## 2. Load Data

In [ ]:
from pathlib import Path

csv_name = "household_data_15min_singleindex.csv"

csv_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
	candidate = root / "data" / csv_name
	if candidate.exists():
		csv_path = candidate
		break

if csv_path is None:
	raise FileNotFoundError(f"Could not find {csv_name} in the current folder or any parent folder.")

df = pd.read_csv(csv_path)
df.head()

## 3. Basic Overview

In [ ]:
print("Shape:", df.shape)
df.info()
df.describe()

## 4. Handle Time

In [ ]:
df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"])
df.set_index("utc_timestamp", inplace=True)

df.head()

## 5. Missing / Interpolated Data

In [ ]:
# Missing values
missing = df.isnull().sum()
print("Missing values:\n", missing[missing > 0])

# Interpolated column
print("\nInterpolated values:")
print(df["interpolated"].value_counts().head())

## 6. Identify Key Columns

In [ ]:
grid_cols = [c for c in df.columns if "grid_import" in c]
pv_cols = [c for c in df.columns if "_pv" in c]

industrial_cols = [c for c in grid_cols if "industrial" in c]
residential_cols = [c for c in grid_cols if "residential" in c]
public_cols = [c for c in grid_cols if "public" in c]

print("Industrial:", len(industrial_cols))
print("Residential:", len(residential_cols))
print("Public:", len(public_cols))

## 7. Create Aggregated Features

In [ ]:
df["industrial_total"] = df[industrial_cols].sum(axis=1)
df["residential_total"] = df[residential_cols].sum(axis=1)
df["public_total"] = df[public_cols].sum(axis=1)

df["total_grid"] = df[grid_cols].sum(axis=1)
df["total_pv"] = df[pv_cols].sum(axis=1)

## 8. Sector Comparison (Daily)

In [ ]:
df[["industrial_total", "residential_total", "public_total"]] \
    .resample("D").mean() \
    .plot(figsize=(12, 5))

plt.title("Daily Average Energy Consumption by Sector")
plt.ylabel("kWh")
plt.show()

## 9. PV vs Grid Usage

In [ ]:
df[["total_grid", "total_pv"]] \
    .resample("D").sum() \
    .plot(figsize=(12, 5))

plt.title("Grid Import vs Solar PV Generation")
plt.ylabel("kWh")
plt.show()

## 10. Hourly Patterns (Peak Usage)

In [ ]:
df["hour"] = df.index.hour

hourly_usage = df.groupby("hour")["total_grid"].mean()

hourly_usage.plot(figsize=(10, 4))
plt.title("Average Energy Usage by Hour")
plt.xlabel("Hour")
plt.ylabel("kWh")
plt.show()

## 11. Top Energy Consumers

In [ ]:
total_usage = df[grid_cols].sum().sort_values(ascending=False)

print("Top 10 consumers:")
print(total_usage.head(10))

## 12. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[grid_cols].corr(), cmap="coolwarm")
plt.title("Correlation Between Households")
plt.show()

## 13. Key Insights

- Industrial sector shows highest energy consumption
- Peak usage occurs in the evening hours
- PV generation reduces grid dependency during daytime
- Some households consume significantly more energy than others
- Interpolated data indicates gaps in measurements